# Data Quality — Full Pipeline (Bronze + Gold)

Runs **all DQ rule sets** against every Bronze table and every Gold table,
writing all results to `hospital_catalog.quality.dq_results`.

| Step | What happens |
|------|--------------|
| 1 | Environment setup and path injection |
| 2 | Schema / table bootstrap (idempotent) |
| 3 | Table configuration — Bronze + Gold |
| 4 | Execute DQ checks |
| 5 | Summary dashboard |

> **Job parameters**: `TABLE_FILTER` (default `ALL`) and `LAYER_FILTER` (default `ALL`, also `bronze` or `gold`)
> are read from Databricks widgets and can be overridden by the job's `base_parameters`.

## 1. Setup

In [ ]:
import sys
from datetime import datetime

sys.path.insert(0, '/Workspace/Users/sallam5100@gmail.com/.bundle/hospital_edw/dev/files')
print('Environment ready.')

In [ ]:
# ── Databricks widgets — allow job base_parameters to override ──────────
# When running in a Databricks job, pass TABLE_FILTER / LAYER_FILTER as
# task base_parameters.  When running interactively, defaults are 'ALL'.
try:
    dbutils.widgets.text('TABLE_FILTER', 'ALL', 'Table filter (key or ALL)')
    dbutils.widgets.text('LAYER_FILTER', 'ALL', 'Layer filter: ALL | bronze | gold')
    TABLE_FILTER = dbutils.widgets.get('TABLE_FILTER')
    LAYER_FILTER = dbutils.widgets.get('LAYER_FILTER')
except Exception:
    # Fall back for unit-testing or non-Databricks environments
    TABLE_FILTER = 'ALL'
    LAYER_FILTER = 'ALL'

print(f'TABLE_FILTER = {TABLE_FILTER!r}')
print(f'LAYER_FILTER = {LAYER_FILTER!r}')

In [ ]:
from config.quality_config import (
    DQ_CATALOG,
    QUALITY_SCHEMA,
    QUARANTINE_SCHEMA,
    BRONZE_SCHEMA,
    DQ_RESULTS_TABLE,
    DQ_RUNS_TABLE,
)
from config.gold_config import GOLD_TABLES

from src.quality.engine import run_dq

# ── Bronze rules ──────────────────────────────────────────────────────────
from src.quality.rules import (
    PATIENT_RULES,
    DOCTOR_RULES,
    DEPARTMENT_RULES,
    ENCOUNTER_RULES,
    ADMISSION_RULES,
    DIAGNOSIS_RULES,
    PROCEDURE_RULES,
    LAB_ORDER_RULES,
    LAB_RESULT_RULES,
    MEDICATION_RULES,
    PRESCRIPTION_RULES,
    PHARMACY_TRANSACTION_RULES,
    INVOICE_RULES,
    PAYMENT_RULES,
    INSURANCE_CLAIM_RULES,
    # ── Gold dimension rules ────────────────────────────────────────────
    DIM_DATE_RULES,
    DIM_DEPARTMENT_RULES,
    DIM_MEDICATION_RULES,
    DIM_DIAGNOSIS_CODE_RULES,
    DIM_PAYER_RULES,
    # ── Gold fact rules ─────────────────────────────────────────────────
    FACT_APPOINTMENTS_RULES,
    FACT_ADMISSIONS_RULES,
    FACT_MEDICAL_RECORDS_RULES,
    FACT_SURGERIES_RULES,
    FACT_LAB_RESULTS_RULES,
    FACT_PRESCRIPTIONS_RULES,
    FACT_PHARMACY_DISPENSES_RULES,
    FACT_BILLING_RULES,
)

print('All imports successful.')

## 2. Schema and Table Bootstrap

Creates the `quality` and `quarantine` schemas and the `dq_results` / `dq_runs`
tracking tables if they do not already exist. Safe to re-run.

In [ ]:
bootstrap_statements = [
    f'CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{QUALITY_SCHEMA}',
    f'CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{QUARANTINE_SCHEMA}',
    f'CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{BRONZE_SCHEMA}',
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULTS_TABLE} (
        dq_run_id        STRING    NOT NULL,
        dq_timestamp     TIMESTAMP NOT NULL,
        table_name       STRING    NOT NULL,
        source_branch    STRING,
        source_system    STRING,
        rule_id          STRING    NOT NULL,
        rule_name        STRING    NOT NULL,
        dq_category      STRING    NOT NULL,
        severity         STRING    NOT NULL,
        column_name      STRING,
        total_records    BIGINT    NOT NULL,
        failed_records   BIGINT    NOT NULL,
        passed_records   BIGINT    NOT NULL,
        failure_rate     DOUBLE,
        status           STRING    NOT NULL,
        rule_description STRING
    ) USING DELTA
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RUNS_TABLE} (
        dq_run_id         STRING    NOT NULL,
        start_timestamp   TIMESTAMP NOT NULL,
        end_timestamp     TIMESTAMP,
        run_status        STRING,
        total_tables      BIGINT,
        total_rules       BIGINT,
        failed_rules      BIGINT,
        critical_failures BIGINT,
        total_records     BIGINT,
        failed_records    BIGINT,
        created_at        TIMESTAMP
    ) USING DELTA
    """,
]

for stmt in bootstrap_statements:
    spark.sql(stmt)

print(f'  OK  {DQ_CATALOG}.{QUALITY_SCHEMA}')
print(f'  OK  {DQ_CATALOG}.{QUARANTINE_SCHEMA}')
print(f'  OK  {DQ_RESULTS_TABLE}')
print(f'  OK  {DQ_RUNS_TABLE}')
print('Bootstrap complete.')

## 3. Table Configuration

Central registry of all tables and their rule sets.
Active set is controlled by `TABLE_FILTER` and `LAYER_FILTER` widgets.

In [ ]:
DQ_TABLE_CONFIG = {

    # ── BRONZE: EMR ───────────────────────────────────────────────────────
    'patients': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.patients',
        'rules': PATIENT_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },
    'doctors': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.doctors',
        'rules': DOCTOR_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },
    'departments': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.departments',
        'rules': DEPARTMENT_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },
    'encounters': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.encounters',
        'rules': ENCOUNTER_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },
    'admissions': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.admissions',
        'rules': ADMISSION_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },
    'diagnoses': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.diagnoses',
        'rules': DIAGNOSIS_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },
    'procedures': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.procedures',
        'rules': PROCEDURE_RULES,
        'source_system': 'EMR',
        'layer': 'bronze',
    },

    # ── BRONZE: Laboratory ─────────────────────────────────────────────────
    'lab_orders': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.lab_orders',
        'rules': LAB_ORDER_RULES,
        'source_system': 'LABORATORY',
        'layer': 'bronze',
    },
    'lab_results': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.lab_results',
        'rules': LAB_RESULT_RULES,
        'source_system': 'LABORATORY',
        'layer': 'bronze',
    },

    # ── BRONZE: Pharmacy ───────────────────────────────────────────────────
    'medications': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.medications',
        'rules': MEDICATION_RULES,
        'source_system': 'PHARMACY',
        'layer': 'bronze',
    },
    'prescriptions': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.prescriptions',
        'rules': PRESCRIPTION_RULES,
        'source_system': 'PHARMACY',
        'layer': 'bronze',
    },
    'pharmacy_transactions': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.pharmacy_transactions',
        'rules': PHARMACY_TRANSACTION_RULES,
        'source_system': 'PHARMACY',
        'layer': 'bronze',
    },

    # ── BRONZE: Billing ────────────────────────────────────────────────────
    'invoices': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.invoices',
        'rules': INVOICE_RULES,
        'source_system': 'BILLING',
        'layer': 'bronze',
    },
    'payments': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.payments',
        'rules': PAYMENT_RULES,
        'source_system': 'BILLING',
        'layer': 'bronze',
    },
    'insurance_claims': {
        'table': f'{DQ_CATALOG}.{BRONZE_SCHEMA}.insurance_claims',
        'rules': INSURANCE_CLAIM_RULES,
        'source_system': 'BILLING',
        'layer': 'bronze',
    },

    # ── GOLD: Dimensions ───────────────────────────────────────────────────
    'gold_dim_date': {
        'table': GOLD_TABLES['dim_date'],
        'rules': DIM_DATE_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_dim_department': {
        'table': GOLD_TABLES['dim_department'],
        'rules': DIM_DEPARTMENT_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_dim_medication': {
        'table': GOLD_TABLES['dim_medication'],
        'rules': DIM_MEDICATION_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_dim_diagnosis_code': {
        'table': GOLD_TABLES['dim_diagnosis_code'],
        'rules': DIM_DIAGNOSIS_CODE_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_dim_payer': {
        'table': GOLD_TABLES['dim_payer'],
        'rules': DIM_PAYER_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },

    # ── GOLD: Facts ────────────────────────────────────────────────────────
    'gold_fact_appointments': {
        'table': GOLD_TABLES['fact_appointments'],
        'rules': FACT_APPOINTMENTS_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_admissions': {
        'table': GOLD_TABLES['fact_admissions'],
        'rules': FACT_ADMISSIONS_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_medical_records': {
        'table': GOLD_TABLES['fact_medical_records'],
        'rules': FACT_MEDICAL_RECORDS_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_surgeries': {
        'table': GOLD_TABLES['fact_surgeries'],
        'rules': FACT_SURGERIES_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_lab_results': {
        'table': GOLD_TABLES['fact_lab_results'],
        'rules': FACT_LAB_RESULTS_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_prescriptions': {
        'table': GOLD_TABLES['fact_prescriptions'],
        'rules': FACT_PRESCRIPTIONS_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_pharmacy_dispenses': {
        'table': GOLD_TABLES['fact_pharmacy_dispenses'],
        'rules': FACT_PHARMACY_DISPENSES_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
    'gold_fact_billing': {
        'table': GOLD_TABLES['fact_billing'],
        'rules': FACT_BILLING_RULES,
        'source_system': 'GOLD',
        'layer': 'gold',
    },
}

# ── Apply filters ─────────────────────────────────────────────────────────
if TABLE_FILTER == 'ALL':
    active = {
        k: v for k, v in DQ_TABLE_CONFIG.items()
        if LAYER_FILTER == 'ALL' or v['layer'] == LAYER_FILTER
    }
else:
    active = {TABLE_FILTER: DQ_TABLE_CONFIG[TABLE_FILTER]}

print(f'Running DQ on {len(active)} tables  '
      f'(TABLE_FILTER={TABLE_FILTER!r}, LAYER_FILTER={LAYER_FILTER!r})')
for k in active:
    print(f'  → {k}')

## 4. Execute DQ Checks

In [ ]:
from pyspark.sql import functions as F

all_results       = []
total_failed_rules = 0

for tbl_key, cfg in active.items():
    tbl_path = cfg['table']
    try:
        df = spark.table(tbl_path)
    except Exception as ex:
        print(f'  ⚠️  SKIP  {tbl_key} — table not found or not yet built: {ex}')
        continue

    result = run_dq(
        spark,
        df,
        table_name    = tbl_key,
        rules         = cfg['rules'],
        source_system = cfg['source_system'],
    )

    icon = '✅' if result['failed_rules'] == 0 else '❌'
    print(
        f"  {icon}  {tbl_key:<37} "
        f"{result['total_records']:>9,} rows  "
        f"{result['rules_executed']:>3} rules  "
        f"{result['failed_rules']:>3} failed"
    )
    all_results.append(result)
    total_failed_rules += result['failed_rules']

print(f'\nDQ run complete — {total_failed_rules} total failed rules across {len(all_results)} tables.')

## 5. Summary Dashboard

In [ ]:
# ── CRITICAL failures — must be zero before going to production ─────────
critical_fails = (
    spark.table(DQ_RESULTS_TABLE)
    .filter((F.col('severity') == 'CRITICAL') & (F.col('status') == 'FAILED'))
    .orderBy(F.desc('failed_records'))
)

n_critical = critical_fails.count()
if n_critical == 0:
    print('✅  No CRITICAL failures — pipeline is production-ready.')
else:
    print(f'❌  {n_critical} CRITICAL rule failures — MUST be fixed before Power BI refresh.')
    display(critical_fails.select(
        'table_name', 'rule_id', 'rule_name', 'column_name',
        'total_records', 'failed_records', 'failure_rate', 'rule_description'
    ))

In [ ]:
# ── Summary by layer, source_system, severity ────────────────────────────
display(
    spark.table(DQ_RESULTS_TABLE)
    .groupBy('source_system', 'severity', 'status')
    .agg(
        F.count('*').alias('rule_count'),
        F.sum('failed_records').alias('total_failed_records'),
        F.round(F.avg('failure_rate') * 100, 2).alias('avg_failure_pct'),
    )
    .orderBy('source_system', 'severity')
)

In [ ]:
# ── Gold Dimension DQ detail ─────────────────────────────────────────────
display(
    spark.table(DQ_RESULTS_TABLE)
    .filter(
        F.col('source_system').isin(['GOLD']) &
        F.col('table_name').startswith('gold_dim')
    )
    .groupBy('table_name', 'dq_category')
    .agg(
        F.count('*').alias('rules'),
        F.sum(F.when(F.col('status') == 'FAILED', 1).otherwise(0)).alias('failed'),
        F.round(F.max('failure_rate') * 100, 2).alias('max_failure_pct'),
    )
    .orderBy('table_name', 'dq_category')
)

In [ ]:
# ── Gold Fact DQ detail ──────────────────────────────────────────────────
display(
    spark.table(DQ_RESULTS_TABLE)
    .filter(
        F.col('source_system').isin(['GOLD']) &
        F.col('table_name').startswith('gold_fact')
    )
    .groupBy('table_name', 'dq_category')
    .agg(
        F.count('*').alias('rules'),
        F.sum(F.when(F.col('status') == 'FAILED', 1).otherwise(0)).alias('failed'),
        F.round(F.max('failure_rate') * 100, 2).alias('max_failure_pct'),
    )
    .orderBy('table_name', 'dq_category')
)

In [ ]:
# ── Top tables by worst failure rate ─────────────────────────────────────
display(
    spark.table(DQ_RESULTS_TABLE)
    .filter(F.col('status') == 'FAILED')
    .groupBy('table_name', 'source_system')
    .agg(
        F.count('*').alias('failed_rules'),
        F.sum('failed_records').alias('total_failed_records'),
        F.round(F.max('failure_rate') * 100, 2).alias('max_rule_failure_pct'),
    )
    .orderBy(F.desc('failed_rules'))
)

In [ ]:
# ── Hard stop for job — raise if CRITICAL failures exist ─────────────────
# This cell causes the Databricks task to FAIL so the job does not proceed
# to the next step with bad data.
if n_critical > 0:
    raise RuntimeError(
        f'DQ CRITICAL failure: {n_critical} CRITICAL rules failed. '
        'Fix upstream data issues before proceeding.'
    )
print('DQ gate passed — no CRITICAL failures.')